# 🎯 Day 03a: Model Selection — Find the Best Model, Scientifically

---

## 🎯 What You'll Master Today
- train_test_split properly
- cross_val_score for reliable evaluation
- GridSearchCV & RandomizedSearchCV
- Learning curves & bias-variance tradeoff

**Goal:** Full model selection workflow from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  MODEL SELECTION — The Scientific Workflow                       ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Step 1: Split → train_test_split (80/20, stratify=y)          ║
║  Step 2: Baseline → DummyClassifier / majority class           ║
║  Step 3: Quick scan → cross_val_score on 3-5 models            ║
║  Step 4: Tune → GridSearchCV / RandomizedSearchCV              ║
║  Step 5: Diagnose → learning_curve (bias vs variance)          ║
║  Step 6: Final → evaluate on held-out test set ONCE            ║
║                                                                   ║
║  ⚠️ NEVER tune on test set. NEVER look at test until final.    ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load data
X, y = load_breast_cancer(return_X_y=True)
print(f"Dataset: {X.shape}, Classes: {np.bincount(y)}")

# ============================================================
# Step 1: Split — stratify keeps class balance
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train balance: {np.bincount(y_train) / len(y_train).round(3)}")
print(f"Test balance:  {np.bincount(y_test) / len(y_test).round(3)}")

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# ============================================================
# Step 2: Baseline — always compare against this
# ============================================================
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_s, y_train)
print(f"\nBaseline (majority class): {dummy.score(X_test_s, y_test):.3f}")

In [ ]:
# ============================================================
# Step 3: Quick Model Scan with cross_val_score
# ============================================================

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

print("Model Comparison (5-fold CV on training data):")
print(f"{'Model':<25} {'Mean Acc':>10} {'Std':>8}")
print("-" * 45)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train_s, y_train, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f"{name:<25} {scores.mean():>10.4f} {scores.std():>8.4f}")

best_model_name = max(results, key=lambda k: results[k].mean())
print(f"\n🏆 Best: {best_model_name}")

In [ ]:
# ============================================================
# Step 4: Hyperparameter Tuning — GridSearchCV
# ============================================================

# Grid search = try ALL combinations (exhaustive)
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto'],
}

grid = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,        # use all CPU cores
    return_train_score=True,
)
grid.fit(X_train_s, y_train)

print(f"GridSearchCV Results:")
print(f"  Best params: {grid.best_params_}")
print(f"  Best CV score: {grid.best_score_:.4f}")
print(f"  Total fits: {len(param_grid['C']) * len(param_grid['kernel']) * len(param_grid['gamma']) * 5}")

# Check if overfitting
best_idx = grid.best_index_
train_score = grid.cv_results_['mean_train_score'][best_idx]
val_score = grid.cv_results_['mean_test_score'][best_idx]
print(f"  Train score: {train_score:.4f} | Val score: {val_score:.4f} | Gap: {train_score - val_score:.4f}")

In [ ]:
# ============================================================
# Step 4b: RandomizedSearchCV — when grid is too large
# ============================================================
from scipy.stats import uniform, randint

param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=50,       # try 50 random combos (vs all combos in Grid)
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
)
random_search.fit(X_train_s, y_train)

print(f"RandomizedSearchCV Results:")
print(f"  Best params: {random_search.best_params_}")
print(f"  Best CV score: {random_search.best_score_:.4f}")
print(f"  Only tried 50 combos (vs millions possible)")

print("\n💡 GridSearch = small param space. RandomizedSearch = large param space.")

In [ ]:
# ============================================================
# Step 5: Learning Curves — Bias vs Variance Diagnosis
# ============================================================

train_sizes, train_scores, val_scores = learning_curve(
    grid.best_estimator_, X_train_s, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='accuracy', n_jobs=-1
)

print("Learning Curve Analysis:")
print(f"{'Samples':>10} {'Train Acc':>12} {'Val Acc':>12} {'Gap':>10} {'Diagnosis':>15}")
print("-" * 62)

for i in range(0, len(train_sizes), 2):  # Print every other
    t_mean = train_scores[i].mean()
    v_mean = val_scores[i].mean()
    gap = t_mean - v_mean
    diagnosis = 'overfitting' if gap > 0.05 else ('underfitting' if v_mean < 0.90 else 'good')
    print(f"{train_sizes[i]:>10d} {t_mean:>12.4f} {v_mean:>12.4f} {gap:>10.4f} {diagnosis:>15}")

print("\n💡 High gap = overfitting (need more data or simpler model)")
print("   Both low = underfitting (need more complex model)")
print("   Both high, small gap = sweet spot")

In [ ]:
# ============================================================
# Step 6: Final Evaluation — test set, ONCE
# ============================================================

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_s)

print("="*50)
print("FINAL TEST SET EVALUATION")
print("="*50)
print(f"Model: SVC{grid.best_params_}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['malignant', 'benign'])}")

print("⚠️ This test score is your FINAL reported number.")
print("   You do NOT tune further based on this.")

---

## 🗺️ Model Selection Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  MODEL SELECTION — Decision Guide                                ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  WORKFLOW:                                                       ║
║    Split → Baseline → CV scan → Tune → Diagnose → Final        ║
║                                                                   ║
║  CROSS-VALIDATION:                                               ║
║    cross_val_score    → quick comparison                        ║
║    StratifiedKFold    → imbalanced data                         ║
║    TimeSeriesSplit    → temporal data                            ║
║                                                                   ║
║  TUNING:                                                         ║
║    GridSearchCV       → small param space (exhaustive)          ║
║    RandomizedSearchCV → large param space (sampling)            ║
║                                                                   ║
║  DIAGNOSIS:                                                      ║
║    Overfit → more data, regularize, simpler model               ║
║    Underfit → more features, complex model, less regularize     ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Why use cross-validation? | Single split is noisy. CV gives mean ± std across folds — reliable estimate |
| 2 | Grid vs Randomized search? | Grid: exhaustive, small space. Random: samples n_iter combos, scales to large spaces |
| 3 | How to detect overfitting? | Train acc >> Val acc, or learning curve gap. Reduce model complexity or add regularization |
| 4 | When to use StratifiedKFold? | Imbalanced classes — ensures each fold has same class distribution |
| 5 | Can you tune on test set? | NEVER. Test set = final report only. Tuning on it gives biased (optimistic) results |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Always start with a DummyClassifier baseline         │
## │  • cross_val_score for quick model comparison            │
## │  • GridSearchCV (small) vs RandomizedSearchCV (large)   │
## │  • learning_curve() diagnoses bias/variance             │
## │  • Test set = final evaluation ONLY                     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **sklearn Pipelines** — build production-ready ML pipelines